# Godzilla Piano Tokenization (Kaggle, resumable)

This notebook tokenizes the full Godzilla piano subset into `.npz` files using triplet tokens `[delta, pitch, duration]`.

Key features:
- deterministic member indexing and resume checkpointing
- periodic metadata persistence (`checkpoint.json`, `manifest.jsonl`, `manifest.json`)
- optional remote sync to Hugging Face dataset repo (recommended for robust resume across sessions)
- optional Kaggle dataset publish snapshots (works, but can be slower/heavier)

Run top-to-bottom. To resume in a new session, keep the same config and rerun.

In [ ]:
import subprocess
import sys

def pip_install(packages):
    cmd = [sys.executable, "-m", "pip", "install"] + list(packages)
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("return code:", result.returncode)
    print("stdout tail:")
    print(((result.stdout or "")[-1200:]) or "<empty>")
    print("stderr tail:")
    print(((result.stderr or "")[-1200:]) or "<empty>")
    return result.returncode == 0

_ = pip_install(["symusic", "huggingface_hub", "kaggle"])


In [ ]:
from pathlib import Path
import os

# Source tar locations to try (first existing path is used).
SOURCE_TAR_CANDIDATES = [
    "/kaggle/input/datasets/projectlosangeles/godzilla-midi-dataset/Godzilla-Piano-MIDI-Dataset-CC-BY-NC-SA.tar.gz",
    "/kaggle/input/godzilla-midi-dataset/Godzilla-Piano-MIDI-Dataset-CC-BY-NC-SA.tar.gz",
    "/kaggle/input/datasets/chickaboomcmurtrie/godzilla-midi-dataset/Godzilla-Piano-MIDI-Dataset-CC-BY-NC-SA.tar.gz",
]

# Local working layout.
WORK_ROOT = Path("/kaggle/working/godzilla_piano_tokenizer")
OUTPUT_ROOT = WORK_ROOT / "tokenized"
DATA_DIR = OUTPUT_ROOT / "data"
META_DIR = OUTPUT_ROOT / "metadata"

# Processing controls.
START_OVER = False
MAX_FILES = 0  # 0 means no cap
MIN_TOKEN_LENGTH = 64 * 3
CHECKPOINT_EVERY_PROCESSED = 1000
UPLOAD_EVERY_ACCEPTED = 500
PROGRESS_EVERY = 200
STOP_AFTER_SECONDS = 0  # 0 means run until done

# Remote backend: "hf" (recommended), "kaggle", or "none".
REMOTE_BACKEND = "hf"

# Hugging Face settings (set as Kaggle secrets for resumable multi-session runs).
HF_OUTPUT_REPO = os.getenv("HF_OUTPUT_REPO", "").strip()
HF_TOKEN = os.getenv("HF_TOKEN", "").strip()
HF_PRIVATE = True
PULL_REMOTE_STATE = True

# Kaggle dataset publish settings (snapshot-style publish; heavier than HF incremental sync).
KAGGLE_DATASET_ID = os.getenv("KAGGLE_DATASET_ID", "").strip()  # username/slug
KAGGLE_DATASET_TITLE = "Godzilla Piano Tokenized"
KAGGLE_PUBLISH_EVERY_ACCEPTED = 5000
KAGGLE_DIR_MODE = "zip"

# Optional: if you attached an older tokenized dataset as Kaggle input, copy its metadata first.
RESUME_FROM_INPUT_DATASET_PATH = ""

print("REMOTE_BACKEND =", REMOTE_BACKEND)
print("OUTPUT_ROOT =", OUTPUT_ROOT)


In [ ]:
import hashlib
import json
import math
import shutil
import tarfile
import time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Set, Tuple

import numpy as np
from symusic import Score


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def md5_text(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def format_eta(seconds: Optional[float]) -> str:
    if seconds is None or not math.isfinite(seconds) or seconds < 0:
        return "unknown"
    sec = int(seconds)
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def retry(desc: str, fn, attempts: int = 3, waits: Sequence[int] = (5, 15, 30)):
    last_exc = None
    for i in range(attempts):
        try:
            return fn()
        except Exception as exc:
            last_exc = exc
            if i == attempts - 1:
                break
            wait = waits[min(i, len(waits) - 1)]
            print(f"{desc} failed ({exc}); retrying in {wait}s ({i + 2}/{attempts})")
            time.sleep(wait)
    raise last_exc


def safe_json_read(path: Path, default):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return default


def safe_json_write(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    tmp.replace(path)


def find_source_tar(candidates: Sequence[str]) -> Path:
    checked = []
    for c in candidates:
        p = Path(c)
        checked.append(str(p))
        if p.exists():
            return p
    raise FileNotFoundError("No source tar found. Checked:\n" + "\n".join(checked))


def copy_resume_metadata_from_input(input_root: str, meta_dir: Path) -> None:
    if not input_root:
        return
    src = Path(input_root)
    if not src.exists():
        print(f"Resume input path does not exist: {src}")
        return

    candidate_meta = [
        src / "metadata",
        src / "tokenized" / "metadata",
    ]
    for m in candidate_meta:
        if m.exists() and m.is_dir():
            for name in ["member_index.json", "checkpoint.json", "manifest.jsonl", "manifest.json"]:
                s = m / name
                d = meta_dir / name
                if s.exists() and not d.exists():
                    d.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(s, d)
                    print(f"Copied resume metadata: {s} -> {d}")
            return


def load_manifest_state(manifest_jsonl_path: Path) -> Tuple[int, Set[int]]:
    if not manifest_jsonl_path.exists():
        return 0, set()

    count = 0
    indices: Set[int] = set()
    with manifest_jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            count += 1
            try:
                row = json.loads(line)
                idx = int(row.get("index", -1))
                if idx >= 0:
                    indices.add(idx)
            except Exception:
                continue
    return count, indices


def append_manifest_entry(manifest_jsonl_path: Path, entry: Dict[str, Any]) -> None:
    manifest_jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_jsonl_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(entry, separators=(",", ":")) + "\n")


def rebuild_manifest_json(manifest_jsonl_path: Path, manifest_json_path: Path) -> int:
    entries: List[Dict[str, Any]] = []
    if manifest_jsonl_path.exists():
        with manifest_jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    entries.append(json.loads(line))
                except Exception:
                    continue
    safe_json_write(manifest_json_path, entries)
    return len(entries)


class SymusicTripletTokenizer:
    """Symusic parser + quantizer matching tokenizer_custom.py bins."""

    DELTA_START = 0
    DELTA_END = 31
    PITCH_START = 32
    PITCH_END = 119
    DUR_START = 120
    DUR_END = 151

    def __init__(self) -> None:
        self.delta_bins = np.concatenate(
            [
                np.asarray([0.0], dtype=np.float64),
                np.logspace(math.log10(1e-4), math.log10(2.0), num=31),
            ],
            axis=0,
        )
        self.duration_bins = np.logspace(
            math.log10(0.05),
            math.log10(4.0),
            num=32,
        ).astype(np.float64)

    @staticmethod
    def _nearest_bin(value: float, bins: np.ndarray) -> int:
        return int(np.argmin(np.abs(bins - float(value))))

    def _quantize_delta(self, delta_seconds: float) -> int:
        clamped = float(max(0.0, min(2.0, delta_seconds)))
        idx = self._nearest_bin(clamped, self.delta_bins)
        return int(self.DELTA_START + idx)

    def _quantize_duration(self, duration_seconds: float) -> int:
        clamped = float(max(0.05, min(4.0, duration_seconds)))
        idx = self._nearest_bin(clamped, self.duration_bins)
        return int(self.DUR_START + idx)

    def _quantize_pitch(self, pitch: int) -> int:
        p = int(max(21, min(108, pitch)))
        return int(self.PITCH_START + (p - 21))

    def parse_events(self, midi_bytes: bytes) -> Optional[List[Tuple[float, int, float]]]:
        score = Score.from_midi(midi_bytes, ttype="second")
        events: List[Tuple[float, int, float]] = []

        if len(score.tracks) == 0:
            return None

        for track in score.tracks:
            program = int(track.program)
            is_drum = bool(track.is_drum)
            if is_drum:
                return None
            if program < 0 or program > 7:
                return None

            for note in track.notes:
                pitch = int(note.pitch)
                if pitch < 21 or pitch > 108:
                    continue
                onset = float(max(0.0, float(note.time)))
                duration = float(max(1e-4, float(note.duration)))
                events.append((onset, pitch, duration))

        events.sort(key=lambda x: (x[0], x[1], x[2]))
        return events

    def encode_events(self, events: List[Tuple[float, int, float]]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        token_ids: List[int] = []
        onset_times: List[float] = []
        durations: List[float] = []
        prev_onset = 0.0

        for onset, pitch, duration in events:
            delta = float(max(0.0, onset - prev_onset))
            prev_onset = onset

            d_tok = self._quantize_delta(delta)
            p_tok = self._quantize_pitch(pitch)
            u_tok = self._quantize_duration(duration)

            token_ids.extend([d_tok, p_tok, u_tok])
            onset_times.extend([float(onset), float(onset), float(onset)])
            durations.extend([float(duration), float(duration), float(duration)])

        return (
            np.asarray(token_ids, dtype=np.int16),
            np.asarray(onset_times, dtype=np.float32),
            np.asarray(durations, dtype=np.float32),
        )


In [ ]:
import subprocess

from huggingface_hub import HfApi, hf_hub_download


def hf_create_repo_if_needed(api: HfApi, repo_id: str, token: str, private: bool = True) -> None:
    retry(
        "Create HF dataset repo",
        lambda: api.create_repo(
            repo_id=repo_id,
            repo_type="dataset",
            private=bool(private),
            exist_ok=True,
            token=token,
        ),
    )


def hf_pull_state_files(api: HfApi, repo_id: str, token: str, output_root: Path, paths: Sequence[str]) -> int:
    pulled = 0
    for rel_path in paths:
        local_target = output_root / rel_path

        def _download_once():
            return hf_hub_download(
                repo_id=repo_id,
                filename=rel_path,
                repo_type="dataset",
                token=token,
            )

        try:
            local_src = retry(f"Download {rel_path}", _download_once, attempts=2, waits=(3, 6))
        except Exception:
            continue

        local_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_src, local_target)
        print(f"Pulled remote state: {rel_path}")
        pulled += 1

    return pulled


def hf_upload_patterns(api: HfApi, repo_id: str, token: str, output_root: Path, allow_patterns: Sequence[str], commit_message: str) -> None:
    patterns = sorted(set(str(p) for p in allow_patterns if str(p).strip()))
    if not patterns:
        return

    retry(
        "HF upload",
        lambda: api.upload_folder(
            folder_path=str(output_root),
            path_in_repo="",
            repo_id=repo_id,
            repo_type="dataset",
            token=token,
            allow_patterns=patterns,
            commit_message=commit_message,
        ),
    )


def ensure_kaggle_metadata(output_root: Path, dataset_id: str, title: str) -> Path:
    meta_path = output_root / "dataset-metadata.json"
    payload = {
        "id": dataset_id,
        "title": title,
        "licenses": [{"name": "CC-BY-NC-SA-4.0"}],
    }
    safe_json_write(meta_path, payload)
    return meta_path


def kaggle_publish_snapshot(output_root: Path, dataset_id: str, title: str, message: str, dir_mode: str = "zip") -> bool:
    if not dataset_id:
        print("Kaggle publish skipped: KAGGLE_DATASET_ID is empty.")
        return False

    ensure_kaggle_metadata(output_root, dataset_id, title)

    version_cmd = ["kaggle", "datasets", "version", "-p", str(output_root), "-m", message, "--dir-mode", dir_mode]
    result = subprocess.run(version_cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("Kaggle dataset version published successfully.")
        return True

    create_cmd = ["kaggle", "datasets", "create", "-p", str(output_root), "--dir-mode", dir_mode]
    result_create = subprocess.run(create_cmd, capture_output=True, text=True)
    if result_create.returncode == 0:
        print("Kaggle dataset created successfully.")
        return True

    print("Kaggle publish failed. version stderr tail:")
    print(((result.stderr or "")[-1200:]) or "<empty>")
    print("Kaggle create stderr tail:")
    print(((result_create.stderr or "")[-1200:]) or "<empty>")
    return False


In [ ]:
if START_OVER and OUTPUT_ROOT.exists():
    print(f"START_OVER=True, deleting previous output: {OUTPUT_ROOT}")
    shutil.rmtree(OUTPUT_ROOT, ignore_errors=True)

for d in [WORK_ROOT, OUTPUT_ROOT, DATA_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

copy_resume_metadata_from_input(RESUME_FROM_INPUT_DATASET_PATH, META_DIR)

source_tar = find_source_tar(SOURCE_TAR_CANDIDATES)
print(f"Source tar: {source_tar}")

api = None
if REMOTE_BACKEND == "hf":
    if not HF_OUTPUT_REPO:
        raise ValueError("REMOTE_BACKEND='hf' requires HF_OUTPUT_REPO (set Kaggle secret).")
    if not HF_TOKEN:
        raise ValueError("REMOTE_BACKEND='hf' requires HF_TOKEN (set Kaggle secret).")

    api = HfApi(token=HF_TOKEN)
    hf_create_repo_if_needed(api, HF_OUTPUT_REPO, HF_TOKEN, private=HF_PRIVATE)

    if PULL_REMOTE_STATE:
        pulled_count = hf_pull_state_files(
            api=api,
            repo_id=HF_OUTPUT_REPO,
            token=HF_TOKEN,
            output_root=OUTPUT_ROOT,
            paths=[
                "metadata/member_index.json",
                "metadata/checkpoint.json",
                "metadata/manifest.jsonl",
                "metadata/manifest.json",
            ],
        )
        print(f"Pulled {pulled_count} state file(s) from HF.")

member_index_path = META_DIR / "member_index.json"
checkpoint_path = META_DIR / "checkpoint.json"
manifest_jsonl_path = META_DIR / "manifest.jsonl"
manifest_json_path = META_DIR / "manifest.json"

member_payload = safe_json_read(member_index_path, default={})
member_names = member_payload.get("member_names") if isinstance(member_payload, dict) else None
if not isinstance(member_names, list) or not member_names:
    print("Building deterministic member index (stream scan)...")
    member_names = []
    with tarfile.open(source_tar, mode="r|gz") as tf:
        for i, member in enumerate(tf, start=1):
            if not member.isfile():
                continue
            name = str(member.name)
            lower = name.lower()
            if lower.endswith(".mid") or lower.endswith(".midi"):
                member_names.append(name)
            if i % 100000 == 0:
                print(f"Scanned {i:,} tar members; kept {len(member_names):,} MIDI entries")

    member_names.sort()
    safe_json_write(
        member_index_path,
        {
            "created_at": utc_now_iso(),
            "source_tar": str(source_tar),
            "total_members": int(len(member_names)),
            "member_names": member_names,
        },
    )
    print(f"Member index built with {len(member_names):,} MIDI entries.")
else:
    print(f"Loaded member index with {len(member_names):,} entries from metadata/member_index.json")

manifest_count, done_indices = load_manifest_state(manifest_jsonl_path)
checkpoint = safe_json_read(
    checkpoint_path,
    default={
        "last_completed_index": -1,
        "accepted": 0,
        "skipped": 0,
        "last_completed_name": "",
        "total_members": int(len(member_names)),
        "recent_completed": [],
        "updated_at": utc_now_iso(),
    },
)

checkpoint.setdefault("recent_completed", [])
checkpoint["total_members"] = int(len(member_names))
checkpoint["accepted"] = int(max(int(checkpoint.get("accepted", 0)), manifest_count))
checkpoint["skipped"] = int(checkpoint.get("skipped", 0))
checkpoint["last_completed_index"] = int(checkpoint.get("last_completed_index", -1))
checkpoint["last_completed_name"] = str(checkpoint.get("last_completed_name", ""))

start_index = max(0, int(checkpoint["last_completed_index"]) + 1)
print("Resume state:")
print(f"  start_index={start_index:,}")
print(f"  accepted={int(checkpoint['accepted']):,}")
print(f"  skipped={int(checkpoint['skipped']):,}")
print(f"  manifest_count={manifest_count:,}")
print(f"  total_members={len(member_names):,}")


In [ ]:
tokenizer = SymusicTripletTokenizer()

accepted = int(checkpoint["accepted"])
skipped = int(checkpoint["skipped"])
last_completed_index = int(checkpoint["last_completed_index"])
last_completed_name = str(checkpoint["last_completed_name"])
recent_completed = checkpoint.get("recent_completed", [])
if not isinstance(recent_completed, list):
    recent_completed = []

pending_upload_paths: List[str] = []
processed_since_checkpoint = 0
session_start = time.time()
next_kaggle_publish_at = accepted + max(1, int(KAGGLE_PUBLISH_EVERY_ACCEPTED))

print("Building tar member lookup map for random access...")
with tarfile.open(source_tar, mode="r:gz") as tar:
    member_lookup = {m.name: m for m in tar.getmembers() if m.isfile()}
print(f"Member lookup ready: {len(member_lookup):,} file entries")

total_members = len(member_names)
if start_index >= total_members:
    print("Nothing to do: start_index is already at or beyond total_members.")

with tarfile.open(source_tar, mode="r:gz") as tar:
    for idx in range(start_index, total_members):
        if MAX_FILES > 0 and accepted >= int(MAX_FILES):
            print(f"MAX_FILES reached ({MAX_FILES}). Stopping.")
            break

        if STOP_AFTER_SECONDS > 0 and (time.time() - session_start) >= float(STOP_AFTER_SECONDS):
            print(f"STOP_AFTER_SECONDS reached ({STOP_AFTER_SECONDS}s). Stopping.")
            break

        name = member_names[idx]
        last_completed_index = idx
        last_completed_name = name
        processed_since_checkpoint += 1

        accepted_entry = False
        entry_npz_path = None
        token_len = 0

        if idx in done_indices:
            continue

        try:
            tar_info = member_lookup.get(name)
            if tar_info is None:
                skipped += 1
            else:
                member_fp = tar.extractfile(tar_info)
                if member_fp is None:
                    skipped += 1
                else:
                    midi_bytes = member_fp.read()
                    events = tokenizer.parse_events(midi_bytes)
                    if events is None:
                        skipped += 1
                    else:
                        tokens, onsets, durations = tokenizer.encode_events(events)
                        token_len = int(tokens.shape[0])
                        if token_len < int(MIN_TOKEN_LENGTH):
                            skipped += 1
                        else:
                            md5 = md5_text(name)
                            rel_path = f"data/{idx:07d}_{md5}.npz"
                            out_npz = OUTPUT_ROOT / rel_path
                            out_npz.parent.mkdir(parents=True, exist_ok=True)
                            np.savez_compressed(
                                out_npz,
                                tokens=tokens,
                                onsets=onsets,
                                durations=durations,
                            )

                            append_manifest_entry(
                                manifest_jsonl_path,
                                {
                                    "index": int(idx),
                                    "md5": md5,
                                    "npz_path": rel_path,
                                    "length": int(token_len),
                                    "source_path": name,
                                },
                            )

                            done_indices.add(idx)
                            accepted += 1
                            accepted_entry = True
                            entry_npz_path = rel_path
                            pending_upload_paths.append(rel_path)
        except Exception as file_exc:
            skipped += 1
            print(f"File error at index {idx} ({name}): {file_exc}")

        recent_completed.append(
            {
                "index": int(idx),
                "name": name,
                "accepted": bool(accepted_entry),
                "npz_path": entry_npz_path,
                "token_length": int(token_len),
            }
        )
        if len(recent_completed) > 64:
            recent_completed = recent_completed[-64:]

        should_checkpoint = (processed_since_checkpoint >= int(CHECKPOINT_EVERY_PROCESSED))
        should_upload = (len(pending_upload_paths) >= int(max(1, UPLOAD_EVERY_ACCEPTED)))

        if (idx + 1) % int(max(1, PROGRESS_EVERY)) == 0:
            elapsed = max(1e-6, time.time() - session_start)
            processed = max(1, idx - start_index + 1)
            rate = processed / elapsed
            remain = max(0, total_members - idx - 1)
            eta = remain / rate if rate > 0 else None
            print(
                f"idx={idx:,}/{total_members:,} accepted={accepted:,} skipped={skipped:,} "
                f"rate={rate:.2f}/s ETA={format_eta(eta)}"
            )

        if should_checkpoint or should_upload:
            checkpoint_payload = {
                "last_completed_index": int(last_completed_index),
                "accepted": int(accepted),
                "skipped": int(skipped),
                "last_completed_name": str(last_completed_name),
                "total_members": int(total_members),
                "recent_completed": recent_completed[-32:],
                "updated_at": utc_now_iso(),
            }
            safe_json_write(checkpoint_path, checkpoint_payload)
            manifest_len = rebuild_manifest_json(manifest_jsonl_path, manifest_json_path)

            if REMOTE_BACKEND == "hf" and pending_upload_paths:
                metadata_patterns = [
                    "metadata/checkpoint.json",
                    "metadata/manifest.jsonl",
                    "metadata/manifest.json",
                    "metadata/member_index.json",
                ]
                hf_upload_patterns(
                    api=api,
                    repo_id=HF_OUTPUT_REPO,
                    token=HF_TOKEN,
                    output_root=OUTPUT_ROOT,
                    allow_patterns=(pending_upload_paths + metadata_patterns),
                    commit_message=(
                        f"Tokenized progress idx={last_completed_index} accepted={accepted} skipped={skipped} manifest={manifest_len}"
                    ),
                )
                pending_upload_paths = []

            if REMOTE_BACKEND == "kaggle" and accepted >= next_kaggle_publish_at:
                _ = kaggle_publish_snapshot(
                    output_root=OUTPUT_ROOT,
                    dataset_id=KAGGLE_DATASET_ID,
                    title=KAGGLE_DATASET_TITLE,
                    message=(
                        f"Tokenized progress idx={last_completed_index} accepted={accepted} skipped={skipped}"
                    ),
                    dir_mode=KAGGLE_DIR_MODE,
                )
                next_kaggle_publish_at = accepted + max(1, int(KAGGLE_PUBLISH_EVERY_ACCEPTED))

            processed_since_checkpoint = 0

# Final flush
checkpoint_payload = {
    "last_completed_index": int(last_completed_index),
    "accepted": int(accepted),
    "skipped": int(skipped),
    "last_completed_name": str(last_completed_name),
    "total_members": int(total_members),
    "recent_completed": recent_completed[-32:],
    "updated_at": utc_now_iso(),
}
safe_json_write(checkpoint_path, checkpoint_payload)
manifest_len = rebuild_manifest_json(manifest_jsonl_path, manifest_json_path)

if REMOTE_BACKEND == "hf":
    metadata_patterns = [
        "metadata/checkpoint.json",
        "metadata/manifest.jsonl",
        "metadata/manifest.json",
        "metadata/member_index.json",
    ]
    hf_upload_patterns(
        api=api,
        repo_id=HF_OUTPUT_REPO,
        token=HF_TOKEN,
        output_root=OUTPUT_ROOT,
        allow_patterns=(pending_upload_paths + metadata_patterns),
        commit_message=(
            f"Tokenization final flush idx={last_completed_index} accepted={accepted} skipped={skipped} manifest={manifest_len}"
        ),
    )

if REMOTE_BACKEND == "kaggle":
    _ = kaggle_publish_snapshot(
        output_root=OUTPUT_ROOT,
        dataset_id=KAGGLE_DATASET_ID,
        title=KAGGLE_DATASET_TITLE,
        message=(
            f"Tokenization final flush idx={last_completed_index} accepted={accepted} skipped={skipped}"
        ),
        dir_mode=KAGGLE_DIR_MODE,
    )

elapsed = time.time() - session_start
print("Done.")
print(f"Elapsed: {elapsed / 60.0:.2f} min")
print(f"Accepted: {accepted:,}")
print(f"Skipped: {skipped:,}")
print(f"Manifest entries: {manifest_len:,}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Manifest: {manifest_json_path}")


## Resume notes

- Re-run this notebook with the same config to continue from `metadata/checkpoint.json`.
- For multi-session reliability, use `REMOTE_BACKEND="hf"` and set `HF_TOKEN` + `HF_OUTPUT_REPO` as Kaggle secrets.
- If using Kaggle backend, each publish is a snapshot-style dataset version and can be slower/heavier than HF incremental sync.
- Outputs are in `OUTPUT_ROOT` with files under `data/` and metadata under `metadata/`.